---
title : "GWDI eenvoudige berekening"
---



Dit notebook laat een minimale GWDI-run zien voor één rastercel (`fid`) via de toolbox-moduleklasse `GwdiInference`.

Het notebook laat het volgende zien:
1. Inlezen van een lokale GWDI-configuratie (`yaml`) voor DataAdapters.
2. Uitvoeren van `GwdiInference.run(...)` met de smoke single-fid dataset.
3. Vergelijken van de uitkomst met de bestaande smoke-baseline uit de unit test.

In [ ]:
import os
from pathlib import Path

from toolbox_continu_inzicht import Config, DataAdapter
from toolbox_continu_inzicht.gwdi import (
    GwdiInference,
    GwdiKnmiRetrieval,
    GwdiWiwbRetrieval,
)

In [ ]:
data_dir = Path.cwd() / "data_sets" / "10.gwdi_simple_calculation"
config_path = data_dir / "gwdi_simple_calculation.yaml"

De configuratie ziet er als volgt uit:

````yaml
GlobalVariables:
    rootdir: "data_sets/10.gwdi_simple_calculation"
    calc_time: "2024-02-15 08:00:00"

    GwdiWiwbRetrieval:
        lag_days: 0
        window_days: 35
        publish_days: 35
        target_date: "2024-02-14"
        wiwb_precipitation_source_code: "Knmi.International.Radar.Composite.Combined"
        resample_frequency: "D"
        resample_period_start: "2024-01-11"
        resample_period_end: "2024-02-14"

    GwdiKnmiRetrieval:
        lag_days: 0
        window_days: 35
        publish_days: 35
        target_date: "2024-02-14"
        resample_frequency: "D"
        resample_period_start: "2024-01-11"
        resample_period_end: "2024-02-14"

DataAdapter:
    gwdi_input_precipitation_static:
        type: csv
        file: "precipitation_single_fid.csv"
    gwdi_input_evaporation_static:
        type: csv
        file: "evaporation_single_fid.csv"

    gwdi_input_precipitation_dynamic:
        type: python
    gwdi_input_evaporation_dynamic:
        type: python

    gwdi_input_climate_sampling_locations:
        type: csv
        file: "gwdi_climate_sampling_locations_single.csv"

    gwdi_input_pastas_mapping:
        type: csv
        file: "gwdi_pastas_mapping_single.csv"
        dtype:
            location: object
            position: object
    gwdi_input_pastas_models:
        type: pastas_models
        path: "pastas_models"
    gwdi_input_stats_minima:
        type: csv
        file: "df_stats_minima_single.csv"
        index_col: 0
    gwdi_output:
        type: python
````

## Eerste voorbeeld: statische klimaatreeks

In dit eerste voorbeeld gebruiken we statische neerslag en verdamping (uit een CSV) om een GWDI-run met `GwdiInference` uit te voeren.

In [ ]:
config = Config(config_path=config_path)
config.lees_config()

data_adapter = DataAdapter(config=config)
module = GwdiInference(data_adapter=data_adapter)

module.run(
    input=[
        "gwdi_input_precipitation_static",
        "gwdi_input_evaporation_static",
        "gwdi_input_pastas_mapping",
        "gwdi_input_pastas_models",
        "gwdi_input_stats_minima",
    ],
    output="gwdi_output",
)

df_gwdi = module.df_out.copy()
df_gwdi

## Tweede voorbeeld: dynamische klimaatreeks via WIWB + KNMI

In dit tweede voorbeeld halen we neerslag en verdamping dynamisch op voor dezelfde GWDI-run:
- `GwdiWiwbRetrieval` levert neerslag (`P`) op.
- `GwdiKnmiRetrieval` levert verdamping (`evaporation`) op.

In `gwdi_simple_calculation.yaml` gebruiken beide retrievals dezelfde resample-configuratie (`resample_frequency`, `resample_period_start`, `resample_period_end`) zodat de tijdstappen op elkaar aansluiten.
Daarna voeren we `GwdiInference` uit met deze dynamische invoer.

In [ ]:
required_env = ("WIWB_CLIENT_ID", "WIWB_KEY", "KNMI_API_KEY")
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    raise UserWarning(
        "Ontbrekende omgevingsvariabelen voor dynamische GWDI-run: "
        + ", ".join(missing_env)
    )
else:
    print(f"OK: de benodigde omgevingsvariabelen zijn aanwezig: {required_env}")

In [ ]:
# Gebruik dezelfde YAML-configuratie als in het statische voorbeeld.
data_adapter_dynamic = DataAdapter(config=config)

wiwb_module = GwdiWiwbRetrieval(data_adapter=data_adapter_dynamic)
wiwb_module.run(
    input="gwdi_input_climate_sampling_locations",
    output="gwdi_input_precipitation_dynamic",
)

knmi_module = GwdiKnmiRetrieval(data_adapter=data_adapter_dynamic)
knmi_module.run(
    input="gwdi_input_climate_sampling_locations",
    output="gwdi_input_evaporation_dynamic",
)

`GwdiInference` vereist exact dezelfde tijdreeksen voor verdamping en neerslag. In de praktijk kan bronbeschikbaarheid per provider toch verschillen. Daarom maken we hieronder expliciet een gezamenlijke (`time`, `fid`)-grid (intersection) voor beide reeksen.

In [ ]:
wiwb_grid = wiwb_module.df_out[["time", "fid"]].drop_duplicates()
knmi_grid = knmi_module.df_out[["time", "fid"]].drop_duplicates()
common_grid = wiwb_grid.merge(knmi_grid, on=["time", "fid"], how="inner")

if len(common_grid) == 0:
    raise UserWarning(
        "Geen overlappende (`time`, `fid`)-grid tussen WIWB en KNMI. "
        "Controleer periode/resample-instellingen."
    )

df_precip_dynamic = wiwb_module.df_out.merge(
    common_grid, on=["time", "fid"], how="inner"
)
df_evap_dynamic = knmi_module.df_out.merge(common_grid, on=["time", "fid"], how="inner")
df_precip_dynamic = df_precip_dynamic.sort_values(["time", "fid"]).reset_index(
    drop=True
)
df_evap_dynamic = df_evap_dynamic.sort_values(["time", "fid"]).reset_index(drop=True)

print("Gezamenlijk grid:")
common_grid

Tenslotte voeren we `GwdiInference` uit met deze dynamische invoer.

In [ ]:
data_adapter_dynamic.set_dataframe_adapter(
    "gwdi_input_precipitation_dynamic",
    df_precip_dynamic,
)
data_adapter_dynamic.set_dataframe_adapter(
    "gwdi_input_evaporation_dynamic",
    df_evap_dynamic,
)

module_dynamic = GwdiInference(data_adapter=data_adapter_dynamic)
module_dynamic.run(
    input=[
        "gwdi_input_precipitation_dynamic",
        "gwdi_input_evaporation_dynamic",
        "gwdi_input_pastas_mapping",
        "gwdi_input_pastas_models",
        "gwdi_input_stats_minima",
    ],
    output="gwdi_output",
)

df_gwdi_dynamic = module_dynamic.df_out.copy()
df_gwdi_dynamic